## Quora Question Pairs Challenge

### **Objective**
The goal of this competition is to predict which of the provided pairs of questions contain two questions with the same meaning. The ground truth is the set of labels that have been supplied by human experts. The ground truth labels are inherently subjective, as the true meaning of sentences can never be known with certainty. Human labeling is also a 'noisy' process, and reasonable people will disagree. As a result, the ground truth labels on this dataset should be taken to be 'informed' but not 100% accurate, and may include incorrect labeling. We believe the labels, on the whole, to represent a reasonable consensus, but this may often not be true on a case by case basis for individual items in the dataset.

### **Procedure**
Implement two solutions:
- ```simple_model```: simple model that will be used as baseline
- ```enhanced_model```: complex model that tries to overcome the limitations of the simple one

### **Data structure**
- id: the id of a training set question pair
- qid1, qid2: unique ids of each question (only available in train.csv)
- question1, question2: the full text of each question
- is_duplicate: the target variable, set to 1 if question1 and question2 have essentially the same meaning, and 0 otherwise.

### **Validation metric used**
train/validation/test results of ROC AUC, as well as precision and recall.

In [62]:
# imports
import pandas as pd
import sklearn
import os
from sklearn import model_selection
import importlib
import numpy as np
from sklearn import *

import utils
importlib.reload(utils)


<module 'utils' from '/home/ntorquet/Documents/llm/TorquetLunaNuria/utils.py'>

In [63]:
# Split data into train, validation and test sets used by both approaches
quora_df = pd.read_csv("data/raw/quora_data.csv")
A_df, test_df = sklearn.model_selection.train_test_split(
    quora_df, test_size=0.05, random_state=123
)
train_df, val_df = sklearn.model_selection.train_test_split(
    A_df, test_size=0.05, random_state=123
)
print("train_df.shape=", train_df.shape)
print("val_df.shape=", val_df.shape)
print("test_df.shape=", test_df.shape)


train_df.shape= (291897, 6)
val_df.shape= (15363, 6)
test_df.shape= (16172, 6)


In [64]:
train_df[20:30]


,id,qid1,qid2,question1,question2,is_duplicate
221406,3341,6623,6624,What do you hate about Toptal's interviewing p...,I need an Indian representative for the supply...,0
225507,132567,82504,212232,What are dimension work?,What dimensions are these?,0
264658,365357,399282,337374,Which is better MacBook or MacBook Pro or MacB...,Is a MacBook Pro worth the money when I can bu...,0
159282,48685,86767,86768,"I wrote a letter, asking for donations from co...","I wrote a letter, asking companies and philant...",1
67672,46012,57994,8120,How do I learn how to invest in stock market a...,What is the best way to learn about stock mark...,1
35861,149668,235708,1672,Who do you love and why?,What do you love? Why?,0
320204,87291,147041,147042,What is currently happening in Turkey?,What's happening in Turkey?,1
259492,183173,110781,38756,Can someone still get your message on snapchat...,If I block someone on snapchat will they still...,0
160754,334925,462157,462158,How long does it take for a human body to comp...,How would it take for a human body to decay to...,0
269562,238681,145570,291752,What is the most inspiring movie you have ever...,What are your most inspiring movies?,1


In [65]:
# store the splits in csv files
train_df.to_csv("data/splits/train_data.csv", index=False)
val_df.to_csv("data/splits/val_data.csv", index=False)
test_df.to_csv("data/splits/test_data.csv", index=False)


### 1. Train a first model

In [66]:
count_vectorizer = sklearn.feature_extraction.text.CountVectorizer(ngram_range=(1, 1))

all_q1 = (
    list(train_df["question1"]) + list(val_df["question1"]) + list(test_df["question1"])
)
all_q2 = (
    list(train_df["question2"]) + list(val_df["question2"]) + list(test_df["question2"])
)
all_questions = all_q1 + all_q2

len(all_questions)


646864

In [67]:
q1_train = utils.cast_list_as_strings(list(train_df["question1"]))
q2_train = utils.cast_list_as_strings(list(train_df["question2"]))
q1_val = utils.cast_list_as_strings(list(val_df["question1"]))
q2_val = utils.cast_list_as_strings(list(val_df["question2"]))
q1_test = utils.cast_list_as_strings(list(test_df["question1"]))
q2_test = utils.cast_list_as_strings(list(test_df["question2"]))


In [68]:
types_ = [type(q).__name__ for q in q1_train]
np.unique(types_)


array(['str'], dtype='<U3')

In [69]:
q1_train[1], q2_train[1]


('How do you convert direct speech into reported speech and vice versa including all cases?',
 "I feel weak at spoken English. I have sentences ready in my mind, but I can't speak it. What should I do?")

Use all the questions in train partition to build a single list ```all_questions``` to fit the ```count_vectorizer```

In [70]:
all_questions = q1_train + q2_train


In [71]:
countr_vectorizer = sklearn.feature_extraction.text.CountVectorizer(ngram_range=(1, 1))
countr_vectorizer.fit(all_questions)


CountVectorizer()

In [73]:
utils.get_features_from_df(train_df, countr_vectorizer)


<291897x149650 sparse matrix of type '<class 'numpy.int64'>'
	with 5861249 stored elements in Compressed Sparse Row format>

In [74]:
X_tr_q1q2 = utils.get_features_from_df(train_df, countr_vectorizer)
X_te_q1q2 = utils.get_features_from_df(test_df, countr_vectorizer)

X_tr_q1q2.shape, X_te_q1q2.shape, test_df.shape, X_te_q1q2.shape

((291897, 149650), (16172, 149650), (16172, 6), (16172, 149650))

In [75]:
logistic = sklearn.linear_model.LogisticRegression(solver="liblinear", random_state=123)
y_train = train_df["is_duplicate"].values
logistic.fit(X_tr_q1q2, y_train)

LogisticRegression(random_state=123, solver='liblinear')